# EPIG Preprocessing: FrameNet + Manual Overrides for Subject vs Context Centric Labeling

One-time offline step to label NRC VAD words as:
- `subject_centric = 1`: internal emotional state (happy, angry, withdrawn...)
- `subject_centric = 0`: external atmosphere/stimulus (dim, chaotic, sunny...)

**Optimizations**:
- Uses smaller v1 lexicon by default (~20k words) for reasonable runtime.
- Includes a manual override dictionary for high-precision emotion words.
- Incremental cache saving.


In [1]:
# Install dependencies
!pip install -q nltk pandas tqdm

import nltk
nltk.download('framenet_v17', quiet=True)
nltk.download('punkt', quiet=True)

from nltk.corpus import framenet as fn
import pandas as pd
import os
import pickle
from tqdm import tqdm
import requests
import zipfile

print("✅ NLTK FrameNet ready!")

✅ NLTK FrameNet ready!


In [3]:
import os
import requests
import zipfile
import pandas as pd

# ====================== DOWNLOAD LEXICON (Fixed) ======================
# Use v1 for speed (~20k words). Change to v2.1 if you want full coverage later.
use_v1 = True

if use_v1:
    vad_url = "https://saifmohammad.com/WebDocs/Lexicons/NRC-VAD-Lexicon.zip"
    zip_path = "NRC-VAD-Lexicon.zip"
    print("Using NRC VAD v1 (~20k words) for faster preprocessing.")
else:
    vad_url = "https://saifmohammad.com/WebDocs/Lexicons/NRC-VAD-Lexicon-v2.1.zip"
    zip_path = "NRC-VAD-Lexicon-v2.1.zip"
    print("Using NRC VAD v2.1 (full ~55k words) — will be slower.")

# Robust download with proper headers
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36"
}

if not os.path.exists(zip_path):
    print("Downloading lexicon...")
    try:
        response = requests.get(vad_url, headers=headers, timeout=60)
        response.raise_for_status()          # Will raise on 4xx/5xx
        with open(zip_path, "wb") as f:
            f.write(response.content)
        print("✅ Download complete.")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error: {e}")
        print("Trying alternative User-Agent...")
        # Fallback with different header
        headers["User-Agent"] = "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)"
        response = requests.get(vad_url, headers=headers, timeout=60)
        response.raise_for_status()
        with open(zip_path, "wb") as f:
            f.write(response.content)
        print("✅ Download successful with fallback User-Agent.")
else:
    print("✅ Lexicon zip already exists.")

# ====================== EXTRACT ======================
extract_dir = "NRC_VAD"
if not os.path.exists(extract_dir):
    print("Extracting...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print("✅ Extraction successful.")
    except zipfile.BadZipFile:
        print("❌ Bad zip file. Please delete the zip and re-run.")
        raise

# ====================== FIND AND LOAD LEXICON FILE ======================
lexicon_path = None
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith('.txt') and ('NRC-VAD' in file.upper() or 'vad' in file.lower()):
            lexicon_path = os.path.join(root, file)
            break
    if lexicon_path:
        break

if not lexicon_path:
    print("Files found in extraction directory:")
    for root, dirs, files in os.walk(extract_dir):
        for f in files:
            print(os.path.join(root, f))
    raise FileNotFoundError("Could not locate the NRC VAD .txt file. Check the printed files above.")

print(f"Loading lexicon from: {lexicon_path}")

vad_df = pd.read_csv(lexicon_path, sep='\t', header=None,
                     names=['Word', 'Valence', 'Arousal', 'Dominance'],
                     encoding='utf-8')

vad_df['Word'] = vad_df['Word'].str.strip().str.lower()
vad_df = vad_df.dropna().drop_duplicates(subset=['Word'])

print(f"✅ Loaded {len(vad_df):,} unique words from NRC VAD Lexicon")
vad_df.head()

Using NRC VAD v1 (~20k words) for faster preprocessing.
✅ Download complete.
Extracting...
✅ Extraction successful.
Loading lexicon from: NRC_VAD/NRC-VAD-Lexicon/NRC-VAD-Lexicon.txt
✅ Loaded 19,970 unique words from NRC VAD Lexicon


,Word,Valence,Arousal,Dominance
0,aaaaaaah,0.479,0.606,0.291
1,aaaah,0.520,0.636,0.282
2,aardvark,0.427,0.490,0.437
3,aback,0.385,0.407,0.288
4,abacus,0.510,0.276,0.485


In [4]:
# ====================== OPTIONAL: LIMIT TO TOP N WORDS ======================
limit_to_top_n = 5000   # Set None or 0 to use all words

if limit_to_top_n and limit_to_top_n > 0 and not use_v1:
    # v2.1 has frequency info in some releases, but for safety we skip sorting for now
    # (v1 does not have frequency column)
    print(f"Limiting to first {limit_to_top_n} words (alphabetical).")
    vad_df = vad_df.head(limit_to_top_n)

# ====================== MANUAL OVERRIDES ======================
# High-precision overrides for common words FrameNet may miss or misclassify
manual_overrides = {
    # subject-centric (internal emotion)
    "happy": 1, "sad": 1, "angry": 1, "joyful": 1, "cheerful": 1, "gloomy": 1,
    "depressed": 1, "excited": 1, "terrified": 1, "scared": 1, "fearful": 1,
    "calm": 1, "peaceful": 1, "anxious": 1, "nervous": 1, "frustrated": 1,
    "smiling": 1, "crying": 1, "laughing": 1, "frowning": 1, "withdrawn": 1,
    "agitated": 1, "content": 1, "ecstatic": 1, "miserable": 1,

    # context-centric (atmosphere) - force 0
    "dim": 0, "bright": 0, "dark": 0, "sunny": 0, "rainy": 0, "chaotic": 0,
    "vibrant": 0, "serene": 0, "stormy": 0, "warm": 0, "cold": 0, "quiet": 0,
    "noisy": 0, "peaceful": 0,   # can be both, but often atmospheric
}

In [5]:
# ====================== FrameNet CLASSIFIER ======================
cache_path = "framenet_subject_cache.pkl"

if os.path.exists(cache_path):
    with open(cache_path, 'rb') as f:
        subject_cache = pickle.load(f)
    print(f"Loaded cache with {len(subject_cache)} entries.")
else:
    subject_cache = {}

EXPERIENCER_INDICATORS = {"Experiencer", "Experiencer_focus", "Experiencer_obj",
                          "Emotion_directed", "Emotions_by_stimulus", "Feeling",
                          "Experiencing", "Mental_stimulus_stimulus_focus"}

def is_subject_centric(word: str) -> int:
    if word in manual_overrides:
        return manual_overrides[word]

    if word in subject_cache:
        return subject_cache[word]

    try:
        frames = []
        for pos in ['v', 'n', 'a']:
            frames.extend(fn.frames_by_lemma(f"{word}.{pos}"))
        if not frames:
            frames.extend(fn.frames_by_lemma(word))

        has_experiencer = False
        for frame in frames:
            frame_name = frame.name.lower()
            if any(ind.lower() in frame_name for ind in ["emotion", "experiencer", "feeling"]):
                has_experiencer = True
                break
            for role in frame.FE.values():
                if role.name in EXPERIENCER_INDICATORS or "experiencer" in role.name.lower():
                    has_experiencer = True
                    break
            if has_experiencer:
                break

        label = 1 if has_experiencer else 0
    except:
        label = 0

    subject_cache[word] = label
    return label

In [6]:
# ====================== RUN LABELING ======================
print("Starting labeling (with manual overrides + FrameNet)...")

labels = []
batch_size = 1000

for i, word in enumerate(tqdm(vad_df['Word'], desc="Processing")):
    label = is_subject_centric(word)
    labels.append(label)

    if (i + 1) % batch_size == 0 or i == len(vad_df) - 1:
        with open(cache_path, 'wb') as f:
            pickle.dump(subject_cache, f)

vad_df['subject_centric'] = labels

# Save final result
output_path = "NRC_VAD_with_subject_centric.csv"
vad_df.to_csv(output_path, index=False)
print(f"\n✅ Saved augmented lexicon to: {output_path}")

Starting labeling (with manual overrides + FrameNet)...


Processing: 100%|██████████| 19970/19970 [15:35<00:00, 21.34it/s]



✅ Saved augmented lexicon to: NRC_VAD_with_subject_centric.csv


In [7]:
# ====================== STATISTICS ======================
total = len(vad_df)
subj = vad_df['subject_centric'].sum()
print("\n" + "="*60)
print("FINAL STATISTICS")
print("="*60)
print(f"Total words          : {total:,}")
print(f"Subject-centric (1)  : {subj:,}  ({subj/total*100:.2f}%)")
print(f"Context-centric (0)  : {total-subj:,}  ({(total-subj)/total*100:.2f}%)")
print("="*60)

print("\nSubject-centric examples:")
print(vad_df[vad_df['subject_centric']==1][['Word','Valence','Arousal']].head(12))

print("\nContext-centric examples:")
print(vad_df[vad_df['subject_centric']==0][['Word','Valence','Arousal']].head(12))


FINAL STATISTICS
Total words          : 19,970
Subject-centric (1)  : 965  (4.83%)
Context-centric (0)  : 19,005  (95.17%)

Subject-centric examples:
           Word  Valence  Arousal
9       abashed    0.177    0.644
24        abhor    0.125    0.602
25   abhorrence    0.167    0.684
29      ability    0.875    0.510
33         able    0.786    0.500
40   abominable    0.120    0.731
41    abominate    0.198    0.788
66       absorb    0.469    0.558
67     absorbed    0.490    0.471
69    absorbing    0.490    0.490
84          aby    0.385    0.452
144         ace    0.643    0.481

Context-centric examples:
           Word  Valence  Arousal
0      aaaaaaah    0.479    0.606
1         aaaah    0.520    0.636
2      aardvark    0.427    0.490
3         aback    0.385    0.407
4        abacus    0.510    0.276
5       abalone    0.500    0.480
6       abandon    0.052    0.519
7     abandoned    0.046    0.481
8   abandonment    0.128    0.430
10        abate    0.255    0.696
11    